In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_colour: str

## Write to state

In [3]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour, 
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]}
        )

In [4]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "google_genai:gemini-3.1-flash-lite",
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [5]:
from langchain.messages import HumanMessage

response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

In [6]:
from pprint import pprint

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='d4c5c906-e9b9-4aa8-a3d7-caa792984c5d'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'update_favourite_colour', 'arguments': '{"favourite_colour": "green"}'}, '__gemini_function_call_thought_signatures__': {'call_4469210': 'EnMKcQFpFH0TRL082845S2y/3JBRotz4wH1TGMnOrEYFat3ed+dRR0Yxe9BWte50H/zUIMobUd8ashKkvfcEHAhcGaWWKHYA67TSxixqqbF3oPcDspVpM0jmAQzzcFZnKzNnH8wlcBOeih+VQYq6Q2gXBBwx'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a14e29-fc85-7df2-8db8-eb115f52b3bb-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'call_4469210', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 69, 'output_tokens': 22, 'total_tokens': 91, 'input_to

In [7]:
response = agent.invoke(
    { 
        "messages": [HumanMessage(content="Hello, how are you?")],
        "favourite_colour": "green"
    },
    {"configurable": {"thread_id": "10"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='b6bcce51-a309-493f-bb6e-d6dab5a89996'),
              AIMessage(content=[{'type': 'text', 'text': "I'm doing well, thank you for asking! How are you doing today?", 'extras': {'signature': 'EnMKcQFpFH0T8uBiufTBFXmkym55qXjHSBGqtjLfzpFLLYHrN3C/5NMRwzpH/8FX0+OPGCzMflkhEJXInVy8Utag2p0rKvbB07l+l1N4tm3AN8U5EsHwkpGt5Ukc1bNxR6R9mC5UrHeiaXQGbW6z9KdHTk2D'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a14e2a-0bc3-7242-b211-c31670293f0a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 70, 'output_tokens': 17, 'total_tokens': 87, 'input_token_details': {'cache_read': 0}})]}


## Read state

In [8]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

agent = create_agent(
    "google_genai:gemini-3.1-flash-lite",
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [9]:
response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='1ef043db-cdcd-4688-9306-398e8ab85a37'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'update_favourite_colour', 'arguments': '{"favourite_colour": "green"}'}, '__gemini_function_call_thought_signatures__': {'call_3006716': 'EnMKcQFpFH0T7gp408W2D7c/wDRK4lDmoBuADrRVYHMRCinFoCDDWKh1j0uQZHPmTcXq/lY9BrpTbcuY+CeK9cNkd5hlqTCVAhB9vz0gu4eXnRovWPJSo944dLJimkGwK5SaNsGjjtBP8HwZoD7spp3mumjf'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a14e2a-31b9-7ea1-891c-5930ebf71c28-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'call_3006716', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 96, 'output_tokens': 22, 'total_tokens': 118, 'input_t

In [10]:
response = agent.invoke(
    { "messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='1ef043db-cdcd-4688-9306-398e8ab85a37'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'update_favourite_colour', 'arguments': '{"favourite_colour": "green"}'}, '__gemini_function_call_thought_signatures__': {'call_3006716': 'EnMKcQFpFH0T7gp408W2D7c/wDRK4lDmoBuADrRVYHMRCinFoCDDWKh1j0uQZHPmTcXq/lY9BrpTbcuY+CeK9cNkd5hlqTCVAhB9vz0gu4eXnRovWPJSo944dLJimkGwK5SaNsGjjtBP8HwZoD7spp3mumjf'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a14e2a-31b9-7ea1-891c-5930ebf71c28-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'call_3006716', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 96, 'output_tokens': 22, 'total_tokens': 118, 'input_t